# SLIIT IT3051 - EV Battery PHM Dual-Task System
## Week 2 - Task 2: Tree-Based Classifiers & Class Imbalance Mitigation (Critical Failure Risk)
### **Assigned Member: Person C**

---
### Overview & Objectives:
Address the 13.45:1 class imbalance using cost-sensitive learning (class_weight="balanced" and scale_pos_weight), Random Forest, and XGBoost.


### 0. Environment Setup & Data Partition Loading
Execute this cell to load the verified preprocessed matrices.


In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Visual aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)

def load_data():
    dataset_path = 'ev battery_failure  Dataset.csv'
    for p in [dataset_path, f'../{dataset_path}', f'../../{dataset_path}']:
        if os.path.exists(p):
            dataset_path = p
            break
            
    df = pd.read_csv(dataset_path)
    
    # 7 Domain Features
    df['temperature_spread'] = df['cell_temperature_max'] - df['cell_temperature_avg']
    df['efficiency_gap'] = (df['charge_efficiency'] - df['discharge_efficiency']).abs()
    df['health_loss_interaction'] = (df['battery_health_percent'] * df['capacity_loss_percent']) / 100.0
    df['resistance_per_1000_cycles'] = (df['internal_resistance'] / (df['cycle_count'] + 1.0)) * 1000.0
    df['c_rate_proxy'] = df['average_charge_power_kw'] / (df['battery_capacity_kwh'] + 1e-5)
    df['cell_voltage_spread'] = df['cell_voltage_std'] / (df['cell_voltage_avg'] + 1e-5)
    df['stress_index'] = df['aggressive_acceleration_score'] * df['hard_braking_score']
    
    id_cols = ['vehicle_id', 'battery_serial']
    target_rul = 'predicted_remaining_life_cycles'
    target_fail = 'battery_failure'
    
    excluded = id_cols + [target_rul, target_fail]
    features = [c for c in df.columns if c not in excluded]
    
    num_cols = df[features].select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = df[features].select_dtypes(include=['object', 'string', 'category']).columns.tolist()
    
    num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
    cat_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
    
    # Task 1 Splits (RUL)
    df_t1 = df[df[target_rul].notnull()].copy()
    X1_tr, X1_te, y1_train, y1_test = train_test_split(df_t1[features], df_t1[target_rul], test_size=0.20, random_state=42)
    ct1 = ColumnTransformer([('num', num_pipe, num_cols), ('cat', cat_pipe, cat_cols)], verbose_feature_names_out=False)
    X1_train = ct1.fit_transform(X1_tr)
    X1_test = ct1.transform(X1_te)
    
    # Task 2 Splits (Failure)
    X2_tr, X2_te, y2_train, y2_test = train_test_split(df[features], df[target_fail], test_size=0.20, random_state=42, stratify=df[target_fail])
    ct2 = ColumnTransformer([('num', num_pipe, num_cols), ('cat', cat_pipe, cat_cols)], verbose_feature_names_out=False)
    X2_train = ct2.fit_transform(X2_tr)
    X2_test = ct2.transform(X2_te)
    
    feature_names = ct1.get_feature_names_out()
    return (X1_train, X1_test, y1_train, y1_test), (X2_train, X2_test, y2_train, y2_test), feature_names

print("Data Loader initialized successfully!")



Data Loader initialized successfully!


In [3]:
_, (X2_train, X2_test, y2_train, y2_test), feature_names = load_data()
print(f'Task 2 Train Shape: {X2_train.shape} | Test Shape: {X2_test.shape}')


Task 2 Train Shape: (16000, 156) | Test Shape: (4000, 156)


### Task 1: Tree Classification Baseline: The Class Imbalance Problem
Train an unweighted Decision Tree and unweighted Random Forest Classifier. Evaluate Precision, Recall, and Confusion Matrix to show how accuracy masks missed failures.


In [7]:
# TODO: Import DecisionTreeClassifier, RandomForestClassifier

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# TODO: Train unweighted DecisionTree and RandomForest

dt_model = DecisionTreeClassifier(random_state=42)

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Train both models
dt_model.fit(X2_train, y2_train)
rf_model.fit(X2_train, y2_train)

# TODO: Display confusion matrices and note the False Negatives

for name, model in [
    ("Decision Tree", dt_model),
    ("Random Forest", rf_model)
]:
    y_pred = model.predict(X2_test)

    print(f"\n{name}")

    print(classification_report(
        y2_test,
        y_pred,
        zero_division=0
    ))

    cm = confusion_matrix(y2_test, y_pred, labels=[0, 1])

    print("Confusion Matrix:")
    print(cm)

    print(f"False Negatives: {cm[1, 0]}")


Decision Tree
              precision    recall  f1-score   support

           0       0.97      0.97      0.97      3723
           1       0.54      0.54      0.54       277

    accuracy                           0.94      4000
   macro avg       0.75      0.75      0.75      4000
weighted avg       0.94      0.94      0.94      4000

Confusion Matrix:
[[3597  126]
 [ 127  150]]
False Negatives: 127

Random Forest
              precision    recall  f1-score   support

           0       0.95      0.99      0.97      3723
           1       0.79      0.34      0.47       277

    accuracy                           0.95      4000
   macro avg       0.87      0.67      0.72      4000
weighted avg       0.94      0.95      0.94      4000

Confusion Matrix:
[[3698   25]
 [ 183   94]]
False Negatives: 183


The baseline Decision Tree achieved 94% accuracy and 54% failure recall, with 127 false negatives. The baseline Random Forest achieved 95% accuracy and 79% failure precision, but its failure recall was only 34%, resulting in 183 false negatives. These results demonstrate that overall accuracy alone is insufficient for evaluating performance on an imbalanced battery failure dataset.

**Analysis & Viva Preparation Notes (Fill this in after running code):**
- What is the Accuracy vs Recall for the unweighted Random Forest?
- How many actual battery failures were missed (False Negatives) by the unweighted model?


### Task 2: Cost-Sensitive Tree Learning (class_weight='balanced')
Train RandomForestClassifier with class_weight='balanced'. Evaluate the trade-off in Precision vs Recall.


In [8]:
# TODO: Train a class-weighted Random Forest

rf_balanced = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

# Train the balanced model
rf_balanced.fit(X2_train, y2_train)

# Predict on the test set
y_pred_balanced = rf_balanced.predict(X2_test)

# Evaluate performance
print("Class-Weighted Random Forest")

print(classification_report(
    y2_test,
    y_pred_balanced,
    zero_division=0
))

cm_balanced = confusion_matrix(
    y2_test,
    y_pred_balanced,
    labels=[0, 1]
)

print("Confusion Matrix:")
print(cm_balanced)

print(f"False Negatives: {cm_balanced[1, 0]}")

Class-Weighted Random Forest
              precision    recall  f1-score   support

           0       0.96      0.98      0.97      3723
           1       0.63      0.51      0.56       277

    accuracy                           0.94      4000
   macro avg       0.79      0.74      0.77      4000
weighted avg       0.94      0.94      0.94      4000

Confusion Matrix:
[[3638   85]
 [ 135  142]]
False Negatives: 135


The class-weighted Random Forest improved failure recall from 34% to 51% and reduced false negatives from 183 to 135 compared with the baseline Random Forest. The failure F1-score increased from 47% to 56%. However, failure precision decreased from 79% to 63%, indicating a trade-off between detecting more failures and producing more false positives.

**Analysis & Viva Preparation Notes (Fill this in after running code):**
- By how much did Recall increase when using class_weight='balanced'?
- What happened to Precision, and why is this an acceptable operational trade-off in battery safety?


### Task 3: Extreme Gradient Boosting with scale_pos_weight
Train HistGradientBoostingClassifier or XGBClassifier with scale_pos_weight = 13.45 (ratio of normal to failure instances).


In [13]:
# TODO: Import XGBClassifier and evaluation metrics

from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score,
    roc_auc_score
)

# TODO: Calculate scale_pos_weight from the training class distribution

negative_count = (y2_train == 0).sum()
positive_count = (y2_train == 1).sum()

scale_weight = negative_count / positive_count

print(f"Scale Pos Weight: {scale_weight:.2f}")


# TODO: Train a cost-sensitive XGBoost classifier

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=scale_weight,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X2_train, y2_train)


# TODO: Generate class predictions and failure probabilities

y_pred_xgb = xgb_model.predict(X2_test)
y_prob_xgb = xgb_model.predict_proba(X2_test)[:, 1]


# TODO: Display classification report and confusion matrix

print("\nCost-Sensitive XGBoost")

print(classification_report(
    y2_test,
    y_pred_xgb,
    zero_division=0
))

cm_xgb = confusion_matrix(
    y2_test,
    y_pred_xgb,
    labels=[0, 1]
)

print("Confusion Matrix:")
print(cm_xgb)

print(f"False Negatives: {cm_xgb[1, 0]}")


# TODO: Evaluate PR-AUC and ROC-AUC

pr_auc_xgb = average_precision_score(
    y2_test,
    y_prob_xgb
)

roc_auc_xgb = roc_auc_score(
    y2_test,
    y_prob_xgb
)

print(f"PR-AUC (Average Precision): {pr_auc_xgb:.4f}")
print(f"ROC-AUC: {roc_auc_xgb:.4f}")

Scale Pos Weight: 13.45

Cost-Sensitive XGBoost
              precision    recall  f1-score   support

           0       0.99      0.92      0.95      3723
           1       0.45      0.91      0.60       277

    accuracy                           0.92      4000
   macro avg       0.72      0.91      0.78      4000
weighted avg       0.96      0.92      0.93      4000

Confusion Matrix:
[[3409  314]
 [  24  253]]
False Negatives: 24
PR-AUC (Average Precision): 0.7420
ROC-AUC: 0.9756


The cost-sensitive XGBoost model was trained using a scale_pos_weight of 13.45 to address the class imbalance. It achieved 91% failure recall, 45% failure precision and a 60% failure F1-score. The model reduced false negatives to 24, although false positives increased to 314. It also achieved a PR-AUC (Average Precision) of 0.7420 and a ROC-AUC of 0.9756. These results demonstrate a substantial improvement in failure detection, with a trade-off in precision.

**Analysis & Viva Preparation Notes (Fill this in after running code):**
- How does cost-sensitive boosting perform compared to balanced Random Forest?
- What are the top failure-driving features identified by the boosting model?


### Task 4: Precision-Recall & ROC Curve Comparison
Plot side-by-side ROC Curves and Precision-Recall Curves comparing all evaluated classification models.


In [ ]:
# TODO: Import ROC and Precision-Recall evaluation metrics

from sklearn.metrics import (
    roc_curve,
    precision_recall_curve,
    roc_auc_score,
    average_precision_score
)

import matplotlib.pyplot as plt
import pandas as pd


# TODO: Collect all trained models

models = {
    "Decision Tree": dt_model,
    "Random Forest": rf_model,
    "Balanced Random Forest": rf_balanced,
    "Cost-Sensitive XGBoost": xgb_model
}


# TODO: Calculate ROC-AUC and PR-AUC for each model

results = []

plt.figure(figsize=(8, 6))

for name, model in models.items():

    y_prob = model.predict_proba(X2_test)[:, 1]

    roc_auc = roc_auc_score(y2_test, y_prob)
    pr_auc = average_precision_score(y2_test, y_prob)

    results.append({
        "Model": name,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc
    })

    fpr, tpr, _ = roc_curve(y2_test, y_prob)

    plt.plot(
        fpr,
        tpr,
        label=f"{name} (AUC={roc_auc:.3f})"
    )


# TODO: Plot ROC curves

plt.plot([0, 1], [0, 1], "k--", label="Random Classifier")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.tight_layout()
plt.show()


# TODO: Plot Precision-Recall curves

plt.figure(figsize=(8, 6))

for name, model in models.items():

    y_prob = model.predict_proba(X2_test)[:, 1]

    precision, recall, _ = precision_recall_curve(
        y2_test,
        y_prob
    )

    pr_auc = average_precision_score(y2_test, y_prob)

    plt.plot(
        recall,
        precision,
        label=f"{name} (AP={pr_auc:.3f})"
    )

# Baseline precision equals the failure prevalence
baseline = (y2_test == 1).mean()

plt.axhline(
    y=baseline,
    color="black",
    linestyle="--",
    label=f"Baseline ({baseline:.3f})"
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve Comparison")
plt.legend()
plt.tight_layout()
plt.show()


# TODO: Display model comparison table

results_df = pd.DataFrame(results)

print("\nModel Performance Comparison:")
print(results_df.to_string(index=False))

**Analysis & Viva Preparation Notes (Fill this in after running code):**
- Why is the PR-AUC curve visually more informative than the ROC curve for a 13.45:1 imbalanced dataset?
- Which model achieved the highest PR-AUC score?
